# SVMModelQ2

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

In [2]:
# Load the cleaned Yelp reviews and three-class sentiment labels.
df = pd.read_csv(DATA_FILE, usecols=["clean_text", "sentiment"])
df = df.dropna(subset=["clean_text", "sentiment"]).copy()
df["clean_text"] = df["clean_text"].astype(str).str.strip()
df = df[df["clean_text"] != ""]

In [3]:
# Use a reproducible stratified 80:20 split to preserve the class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"],
)

In [4]:
# The cleaned text retains negation; bigrams capture phrases such as "not good".
pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
                strip_accents="unicode",
            ),
        ),
        (
            "clf",
            LinearSVC(
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ]
)

In [5]:
# Train and save the baseline three-class Yelp Linear SVM model.
pipeline.fit(X_train, y_train)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / "svm_pipeline.joblib")

['C:\\Users\\yingx\\Desktop\\TextAssignment\\Part_B\\models\\svm_pipeline.joblib']

In [6]:
# Evaluate the baseline model on the unseen test set.
predictions = pipeline.predict(X_test)

print("=== Q2: Yelp Linear SVM Classification Report (3-class sentiment) ===")
print(f"Rows used: {len(df)}")
print("Class distribution:")
print(df["sentiment"].value_counts())
print()
print(classification_report(y_test, predictions, digits=4, zero_division=0))
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

=== Q2: Yelp Linear SVM Classification Report (3-class sentiment) ===
Rows used: 29998
Class distribution:
sentiment
positive    12000
negative    11998
neutral      6000
Name: count, dtype: int64

              precision    recall  f1-score   support

    negative     0.8171    0.8154    0.8163      2400
     neutral     0.4655    0.4500    0.4576      1200
    positive     0.8086    0.8237    0.8161      2400

    accuracy                         0.7457      6000
   macro avg     0.6971    0.6964    0.6967      6000
weighted avg     0.7434    0.7457    0.7445      6000

Accuracy: 0.7457
